In [12]:
# --- STEP 1: DOWNLOAD & EXTRACT ---
!pip install mediapipe==0.10.14
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import cv2, time, os
from google.colab.patches import cv2_imshow
import mediapipe as mp

mp_pose = mp.solutions.pose
teacher = mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5)

print("✅ MediaPipe 0.10.14 installed. Solutions API restored!")

# 1. Input the file from Google Drive (Using the ID you provided)
# This downloads your dataset and names it 'dataset.zip' on the disk
!gdown --id '1LuQvULB4UnZUu_AhgmjUfNK5ZV3IMNiH' -O dataset.zip

# 1. Force-delete the old folder if it exists (removes all old data)
!rm -rf /content/human_data

# 2. Re-create the empty folder
!mkdir -p /content/human_data

# 3. Unzip the new data into the fresh folder
!unzip -o -q "dataset.zip" -d /content/human_data




✅ MediaPipe 0.10.14 installed. Solutions API restored!
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1LuQvULB4UnZUu_AhgmjUfNK5ZV3IMNiH
From (redirected): https://drive.google.com/uc?id=1LuQvULB4UnZUu_AhgmjUfNK5ZV3IMNiH&confirm=t&uuid=c250b198-c33c-4ea6-af5a-f151acd6cd94
To: /content/dataset.zip
100% 272M/272M [00:04<00:00, 63.8MB/s]


In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
# --- STEP 2: BUILD, TRAIN & INFER ---

# When we trained the model, cv2.resize() was used to resize images to (128, 128)
def build_human_detector(input_shape=(128, 128, 3)):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, (3, 3), activation='relu')(inputs)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)

    presence = layers.Dense(1, activation='sigmoid', name='presence')(x)
    location = layers.Dense(4, activation='sigmoid', name='location')(x)

    model = models.Model(inputs=inputs, outputs=[presence, location])
    model.compile(optimizer='adam',
                  loss={'presence': 'binary_crossentropy', 'location': 'mse'},
                  loss_weights={'presence': 1.0, 'location': 5.0})
    return model

def prepare_training_data(base_dir):
    images, presence_labels, box_coords = [], [], []

    # Auto-fix for the nested folder structure we found earlier
    current_path = base_dir
    if not os.path.exists(os.path.join(current_path, '1')):
        subdirs = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
        if subdirs: current_path = os.path.join(base_dir, subdirs[0])

    print(f"Final Input Path: {current_path}")
    categories = {'1': 1.0, '0': 0.0}
    for folder, label in categories.items():
        path = os.path.join(current_path, folder)
        if not os.path.exists(path): continue

        print(f"Processing folder '{folder}'...")
        for img_name in os.listdir(path)[:1000]: # Limit to 1000 for speed
            img = cv2.imread(os.path.join(path, img_name))
            if img is None: continue

            box = [0.0, 0.0, 0.0, 0.0]
            if label == 1.0:
                results = teacher.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
                if results.pose_landmarks:
                    lms = results.pose_landmarks.landmark
                    xs, ys = [l.x for l in lms], [l.y for l in lms]
                    xmin, ymin = max(0, min(xs)), max(0, min(ys))
                    box = [xmin, ymin, min(1, max(xs)) - xmin, min(1, max(ys)) - ymin]
                else: box = [0.2, 0.1, 0.6, 0.8]

            images.append(cv2.resize(img, (128, 128)) / 255.0)
            presence_labels.append(label)
            box_coords.append(box)

    return np.array(images), np.array(presence_labels), np.array(box_coords)

# --- START PROCESS ---
# Updated path to match your successful unzip from Step 1
X_train, y_p, y_l = prepare_training_data('/content/human_data')

my_detector = build_human_detector()
print("🔥 Starting Explicit Training...")
my_detector.fit(X_train, {'presence': y_p, 'location': y_l}, epochs=10, batch_size=32)





Final Input Path: /content/human_data/human detection dataset
Processing folder '1'...
Processing folder '0'...
🔥 Starting Explicit Training...
Epoch 1/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 24s 738ms/step - location_loss: 0.1014 - loss: 2.0197 - presence_loss: 1.5124
Epoch 2/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 40s 720ms/step - location_loss: 0.0589 - loss: 0.9289 - presence_loss: 0.6342
Epoch 3/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 23s 786ms/step - location_loss: 0.0505 - loss: 0.7798 - presence_loss: 0.5274
Epoch 4/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 21s 722ms/step - location_loss: 0.0435 - loss: 0.6441 - presence_loss: 0.4268
Epoch 5/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 41s 720ms/step - location_loss: 0.0393 - loss: 0.5511 - presence_loss: 0.3545
Epoch 6/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 23s 792ms/step - location_loss: 0.0255 - loss: 0.3604 - presence_loss: 0.2331
Epoch 7/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 41s 804ms/step - location_loss: 0.0170 - loss: 0.2513 - presence_loss: 0.1661
Epoch 8/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 21s 73

In [15]:
# --- STEP 4: FINAL VIDEO INFERENCE ---

# 1. Open the video you uploaded (ensure it's named 'input.mp4')
cap = cv2.VideoCapture('input.mp4')
w, h = int(cap.get(3)), int(cap.get(4))
fps = cap.get(cv2.CAP_PROP_FPS) if cap.get(cv2.CAP_PROP_FPS) > 0 else 24
out = cv2.VideoWriter('output.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

if not cap.isOpened():
    print("❌ Error: 'input.mp4' not found. Please upload it to the sidebar!")
else:
    print("🎥 Detecting humans... Processing frames...")
    while cap.isOpened():
        success, frame = cap.read()
        if not success: break

        # Prepare frame for CNN (128x128, normalized)
        img_input = np.expand_dims(cv2.resize(frame, (128, 128)) / 255.0, axis=0)

        # Model returns [presence_array, location_array]
        preds = my_detector.predict(img_input, verbose=0)

        # Correctly extract values from the multi-output list
        # presence is index 0, coords is index 1
        conf = float(preds[0].flatten()[0])
        coords = preds[1].flatten()

        if conf > 0.5:
            # Model learned [xmin, ymin, width, height]
            nx, ny, nw, nh = coords

            # Convert normalized (0-1) to pixel coordinates
            x1, y1 = int(nx * w), int(ny * h)
            x2 = int((nx + nw) * w)
            y2 = int((ny + nh) * h)

            # Draw the EXPLICIT green box
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)
            cv2.putText(frame, f"Human: {conf*100:.1f}%", (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        out.write(frame)

    cap.release()
    out.release()
    print("✅ Success! Right-click 'output.mp4' in the sidebar and select 'Download'.")


🎥 Detecting humans... Processing frames...
✅ Success! Right-click 'output.mp4' in the sidebar and select 'Download'.
